# **Базовые скрипты для моделирования временных рядов**

In [101]:
import yfinance as yf

import numpy as np

import datetime as dt
import pandas as pd

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import BaggingRegressor
from sklearn.base import BaseEstimator

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from typing import Union, Tuple, Dict, Optional, Any

import warnings
warnings.filterwarnings('ignore')

In [102]:
TEST_SIZE = 0.05
SEED = 42

## **Part 1. TimeSeriesOverview**

### **Загрузка данных**

In [103]:
def data_stocks_load(
    ticker: str,
    start_date: str,
    end_date: str
    ) -> pd.DataFrame:
    return yf.download(ticker, start=start_date, end=end_date)



def data_to_timeseries(
    data: pd.DataFrame
    ) -> pd.DataFrame:
    data.reset_index(inplace=True)
    timeseries = pd.DataFrame()
    timeseries['Date'] = data['Date']
    timeseries['Close'] = data['Close']
    timeseries = timeseries.set_index('Date')

    return timeseries


In [104]:
MAANG_tickers = {
    'META': 'Meta',
    'AAPL': 'Apple',
    'AMZN': 'Amazon',
    'NFLX': 'Netflix',
    'GOOGL': 'Google'
}


start_date = '2022-06-01' # формат: 'yyyy-mm-dd'
end_date = dt.date.today() + dt.timedelta(days=1)


ticker = list(MAANG_tickers.keys())[-1]
stocks = data_stocks_load(ticker, start_date, end_date)

[*********************100%***********************]  1 of 1 completed


In [105]:
raw_TS = data_to_timeseries(stocks)

### **Валидация входных данных**

In [106]:
def timeseries_missing_fill(
    timeseries: pd.DataFrame
    ) -> pd.DataFrame:
    full_index = pd.date_range(start=timeseries.index.min(), end=timeseries.index.max(), freq='D')
    timeseries = timeseries.reindex(full_index)
    timeseries['Close'] = timeseries['Close'].interpolate(method='time')

    return timeseries

In [107]:
TS = timeseries_missing_fill(raw_TS)

### **Визуализация**

In [108]:
def timeseries_visualization(
    timeseries: pd.DataFrame,
    smoothing: int=1
    ) -> None:
    if smoothing == 1:
        fig = px.line(timeseries, x=timeseries.index, y=timeseries['Close'], title=f"{MAANG_tickers[ticker]}'s stocks")
    else:
        fig = px.line(title=f"{MAANG_tickers[ticker]}'s stocks")
        fig.add_scatter(x=timeseries.index, y=timeseries['Close'], mode='lines', name='Original stocks', line=dict(color='silver'))
        fig.add_scatter(x=timeseries.index, y=timeseries.rolling(window=smoothing).mean()['Close'], mode='lines', name=f'SMA, window={smoothing}', line=dict(color='red'))

    fig.update_layout(template='plotly_white', width=1000, height=500)
    fig.update_xaxes(title_text="Date")
    fig.update_yaxes(title_text="Close")
    fig.show()


In [109]:
timeseries_visualization(TS)

## **Part 2. TimeSeriesAnalytics**

### **Сглаживание**

In [110]:
def timeseries_sma_smoothing_visualization(
    timeseries: pd.DataFrame,
    *smoothings_values: int
    ) -> Union[None, str]:
    len_smoothing_values = len(smoothings_values)

    fig = px.line(title=f"{MAANG_tickers[ticker]}'s stocks with SMA smoothing")
    if len_smoothing_values == 1:
        fig.add_scatter(x=timeseries.index, y=timeseries['Close'], mode='lines', name='Original stocks', line=dict(color='silver'))
        fig.add_scatter(x=timeseries.index, y=timeseries.rolling(window=smoothings_values[0]).mean()['Close'], mode='lines', name=f'SMA, window={smoothings_values[0]}', line=dict(color='yellow'))
    elif len_smoothing_values == 2:
        fig.add_scatter(x=timeseries.index, y=timeseries['Close'], mode='lines', name='Original stocks', line=dict(color='silver'))
        fig.add_scatter(x=timeseries.index, y=timeseries.rolling(window=smoothings_values[0]).mean()['Close'], mode='lines', name=f'SMA, window={smoothings_values[0]}', line=dict(color='yellow'))
        fig.add_scatter(x=timeseries.index, y=timeseries.rolling(window=smoothings_values[1]).mean()['Close'], mode='lines', name=f'SMA, window={smoothings_values[1]}', line=dict(color='orange'))
    elif len_smoothing_values == 3:
        fig.add_scatter(x=timeseries.index, y=timeseries['Close'], mode='lines', name='Original stocks', line=dict(color='silver'))
        fig.add_scatter(x=timeseries.index, y=timeseries.rolling(window=smoothings_values[0]).mean()['Close'], mode='lines', name=f'SMA, window={smoothings_values[0]}', line=dict(color='yellow'))
        fig.add_scatter(x=timeseries.index, y=timeseries.rolling(window=smoothings_values[1]).mean()['Close'], mode='lines', name=f'SMA, window={smoothings_values[1]}', line=dict(color='orange'))
        fig.add_scatter(x=timeseries.index, y=timeseries.rolling(window=smoothings_values[2]).mean()['Close'], mode='lines', name=f'SMA, window={smoothings_values[2]}', line=dict(color='red'))
    else:
        return "Есть ограничение на количество сглаживаний! Вводите от 1 до 3 параметров для размеров окна"

    fig.update_layout(template='plotly_white', width=1000, height=500)
    fig.update_xaxes(title_text="Date")
    fig.update_yaxes(title_text="Close")
    fig.show()

In [111]:
timeseries_sma_smoothing_visualization(TS, 7, 30, 90)

### **Декомпозиция**

In [112]:
def timeseries_decompose(
    timeseries: pd.DataFrame,
    model_type: str='multiplicative',
    period_value: int=30
    ) -> None:
    timeseries.index = pd.to_datetime(timeseries.index)

    decompose = seasonal_decompose(timeseries['Close'], model=model_type, period=period_value)
    decompose_observed = decompose.observed
    decompose_trend = decompose.trend
    decompose_seasonal = decompose.seasonal
    decompose_resid = decompose.resid

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=('Observed', 'Trend', 'Seasonal', 'Residual')
    )

    fig.add_trace(go.Scatter(
        x=decompose_observed.index,
        y=decompose_observed,
        mode='lines',
        name='Observed',
        line=dict(color='blue', width=1.5)),
        row=1, col=1)

    fig.add_trace(go.Scatter(
        x=decompose_trend.index,
        y=decompose_trend,
        mode='lines',
        name='Trend',
        line=dict(color='red', width=1.5)),
        row=2, col=1)

    fig.add_trace(go.Scatter(
        x=decompose_seasonal.index,
        y=decompose_seasonal,
        mode='lines',
        name='Seasonal',
        line=dict(color='green', width=1.5)),
        row=3, col=1)

    fig.add_trace(go.Scatter(
        x=decompose_resid.index,
        y=decompose_resid,
        mode='markers',
        name='Residual',
        marker=dict(color='black', size=2, opacity=0.5)),
        row=4, col=1)

    fig.update_layout(
        width=1000,
        height=600,
        title_text='Time series decomposition',
        showlegend=False,
        template='plotly_white'
    )

    fig.update_yaxes(title_text="Close", row=1, col=1)
    fig.update_yaxes(title_text="Close", row=2, col=1)
    fig.update_yaxes(title_text="Close", row=3, col=1)
    fig.update_yaxes(title_text="Close", row=4, col=1)
    fig.update_xaxes(title_text="Data", row=4, col=1)

    fig.show()


In [113]:
model_types = ['additive', 'multiplicative']
period_values = [7, 30, 90]

timeseries_decompose(TS)

### **Автокорреляция**

In [114]:
def timeseries_autocorrelation(
    timeseries: pd.DataFrame,
    lags: int=100
    ) -> Tuple[pd.DataFrame, str]:
    acorr_ljungbox_df = acorr_ljungbox(timeseries['Close'], return_df=True, lags=lags)
    len_acorr_ljungbox_df = acorr_ljungbox_df[acorr_ljungbox_df['lb_pvalue'] >= 0.05].shape[0]

    if len_acorr_ljungbox_df == 0:
        return acorr_ljungbox_df, "Автокорреляция присутствует на всех проверяемых лагах"
    else:
        return acorr_ljungbox_df, "Автокорреляция отсутствует на некоторых лагах"

In [115]:
ac_lb_df, ac_lb_result = timeseries_autocorrelation(TS)

ac_lb_result

'Автокорреляция присутствует на всех проверяемых лагах'

### **Стационарность**

In [116]:
def adf_test(
    timeseries: pd.Series
    ) -> None:
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "Lags Used",
            "Number of Observations Used",
        ]
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)


def kpss_test(
    timeseries: pd.Series
    ) -> None:
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto")
    kpss_output = pd.Series(
        kpsstest[0:3],
        index=[
            "Test Statistic",
            "p-value",
            "Lags Used"
        ]
    )
    for key, value in kpsstest[3].items():
        kpss_output["Critical Value (%s)" % key] = value
    print(kpss_output)


def timeseries_stationarity(
    timeseries: pd.Series
    ) -> str:
    df_test = adfuller(timeseries, autolag="AIC")
    kpss_test = kpss(timeseries, regression="c", nlags="auto")

    df_p_value = round(df_test[1], 5)
    kpss_p_value = round(kpss_test[1], 5)

    if (df_p_value <= 0.05) and (kpss_p_value > 0.05):
        return "Временной ряд стационарный"
    elif (df_p_value > 0.05) and (kpss_p_value <= 0.05):
        return "Временной ряд нестационарный"
    elif (df_p_value <= 0.05) and (kpss_p_value <= 0.05):
        return "Результаты противоречивы"
    elif (df_p_value > 0.05) and (kpss_p_value > 0.05):
        return "Требуется больше данных"

In [117]:
timeseries_stationarity(TS)

'Временной ряд нестационарный'

## **Part 3. TimeSeriesForecasting**

### **Инженерия признаков**

In [118]:
def code_mean(
    data: pd.DataFrame,
    time_feature: str,
    real_feature: str
    ) -> Dict[Any, float]:
    return dict(data.groupby(time_feature)[real_feature].mean())


def code_std(
    data: pd.DataFrame,
    time_feature: str,
    real_feature: str
    ) -> Dict[Any, float]:
    return dict(data.groupby(time_feature)[real_feature].std(ddof=1))


def timeseries_preprocessing_to_dataset_with_train_test_split(
    timeseries: pd.DataFrame,
    test_size: float=0.2,
    lag_begin: int=1,
    lag_end: int=7
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    timeseries = pd.DataFrame(timeseries.copy())

    # считаем индекс в датафрейме, после которого начинается тестовый отрезок
    test_index = int(len(timeseries) * (1 - test_size))

    # добавляем лаги исходного ряда в качестве признаков
    for i in range(lag_begin, lag_end + 1):
        timeseries[f"lag_{i}"] = timeseries.Close.shift(i)

    timeseries["weekday"] = timeseries.index.weekday
    # timeseries["week"] = timeseries.index.week
    timeseries["month"] = timeseries.index.month
    timeseries["year"] = timeseries.index.year

    # считаем средние только по тренировочной части, чтобы избежать лика (data leak)
    timeseries["weekday_mean"] = list(map(code_mean(timeseries[:test_index], "weekday", "Close").get, timeseries.weekday))
    # timeseries["week_mean"] = list(map(code_mean(timeseries[:test_index], "week", "Close").get, timeseries.week))
    timeseries["month_mean"] = list(map(code_mean(timeseries[:test_index], "month", "Close").get, timeseries.month))
    timeseries["year_mean"] = list(map(code_mean(timeseries[:test_index], "year", "Close").get, timeseries.year))

    # считаем отклонения только по тренировочной части, чтобы избежать лика (data leak)
    timeseries["weekday_std"] = list(map(code_std(timeseries[:test_index], "weekday", "Close").get, timeseries.weekday))
    # timeseries["week_std"] = list(map(code_std(timeseries[:test_index], "week", "Close").get, timeseries.week))
    timeseries["month_std"] = list(map(code_std(timeseries[:test_index], "month", "Close").get, timeseries.month))
    timeseries["year_std"] = list(map(code_std(timeseries[:test_index], "year", "Close").get, timeseries.year))

    # выкидываем закодированные средними признаки
    timeseries.drop(["weekday", 'month', 'year'], axis=1, inplace=True)

    timeseries = timeseries.dropna()
    timeseries = timeseries.reset_index(drop=True)

    # разбиваем весь датасет на тренировочную и тестовую выборку
    X_train = timeseries.loc[:test_index].drop(["Close"], axis=1)
    y_train = timeseries.loc[:test_index]["Close"]
    X_test = timeseries.loc[test_index:].drop(["Close"], axis=1)
    y_test = timeseries.loc[test_index:]["Close"]

    return X_train, X_test, y_train, y_test

In [119]:
X_train, X_test, y_train, y_test = timeseries_preprocessing_to_dataset_with_train_test_split(TS, test_size=TEST_SIZE)

### **Выбор, обучение и оценка качества модели**

In [120]:
def timeseries_model_training_and_prediction(
    model: BaseEstimator,
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: Union[pd.Series, np.ndarray]
    ) -> Union[pd.Series, np.ndarray]:
    model.fit(X_train, y_train)
    y_pred = pd.Series(model.predict(X_test))

    return y_pred


def symmetric_mean_absolute_percentage_error(
    y_test: Union[pd.Series, np.ndarray],
    y_pred: Union[pd.Series, np.ndarray]
    ) -> float:
    y_test = np.asarray(y_test)
    y_pred = np.asarray(y_pred)

    denominator = (np.abs(y_test) + np.abs(y_pred)) / 2
    mask = denominator != 0

    if not np.any(mask):
        return 0.0  # все значения нулевые – ошибка 0

    return np.mean(np.abs(y_test[mask] - y_pred[mask]) / denominator[mask])


def timeseries_model_metrics(
    y_test: Union[pd.Series, np.ndarray],
    y_pred: Union[pd.Series, np.ndarray]
    ) -> Tuple[Tuple[str], Tuple[float]]:
    R2 = round(r2_score(y_test, y_pred), 5)
    RMSE = round(root_mean_squared_error(y_test, y_pred), 5)
    MAE = round(mean_absolute_error(y_test, y_pred), 5)
    MAPE = round(mean_absolute_percentage_error(y_test, y_pred) * 100, 5)
    SMAPE = round(symmetric_mean_absolute_percentage_error(y_test, y_pred)  * 100, 5)

    return ('R2', 'RMSE', 'MAE', 'MAPE', 'SMAPE'), (R2, RMSE, MAE, MAPE, SMAPE)

In [121]:
ML_models_dict = {
    'baseline': LinearRegression(),
    'general': BaggingRegressor(
        estimator=Ridge(alpha=1.0, solver='auto'),  # базовая модель
        n_estimators=100,                           # количество моделей в ансамбле
        max_samples=0.8,                            # размер подвыборки (80% данных)
        bootstrap=True,                             # выборка с возвращением
        bootstrap_features=False,                   # не использовать бутстрап по признакам
        n_jobs=-1,                                  # параллельное обучение
        random_state=SEED,
    )
}

In [122]:
y_predictions = timeseries_model_training_and_prediction(ML_models_dict['baseline'], X_train, X_test, y_train)
metrics_results = timeseries_model_metrics(y_test, y_predictions)

### **Прогноз на Х единиц времени вперёд**

In [123]:
def timeseries_forecasting_visualization(
    y_train: Union[pd.Series, np.ndarray],
    y_test: Union[pd.Series, np.ndarray],
    y_pred: Union[pd.Series, np.ndarray]
    ) -> None:
    fig = px.line(title=f"{MAANG_tickers[ticker]}'s stocks forecasting")
    fig.add_scatter(x=y_train.index, y=y_train, mode='lines', name='y_train', line=dict(color='blue'))
    fig.add_scatter(x=y_test.index, y=y_test, mode='lines', name='y_test', line=dict(color='green'))
    fig.add_scatter(x=y_test.index, y=y_pred, mode='lines', name='y_pred', line=dict(color='red'))

    fig.update_layout(template='plotly_white', width=1000, height=500)
    fig.update_xaxes(title_text="Index")
    fig.update_yaxes(title_text="Values")
    fig.show()

In [124]:
timeseries_forecasting_visualization(y_train, y_test, y_predictions)

In [125]:
model_report = pd.DataFrame({
    'metrics': metrics_results[0],
    'values': metrics_results[1],
}).set_index('metrics')

model_report

,values
metrics,
R2,0.89005
RMSE,6.42131
MAE,3.96009
MAPE,1.06290
SMAPE,1.06759


# **Локальный запуск сервиса**

## **C:\\\..\TimeSeriesAnalysis**
```
pip install -r requirements.txt
```

## **Terminal 1**
```
cd ..\TimeSeriesAnalysis\backend
uvicorn main:app --reload
```

## **Terminal 2**
```
cd ..\TimeSeriesAnalysis\frontend
streamlit run app.py
```

